In [1]:
import os
import uuid
import shutil
#import pylatex
import pandas as pd
import lightkurve as lk 
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import seaborn as sns

from datetime import datetime
from tqdm.notebook import tqdm
from multiprocessing import cpu_count, Pool
#from pylatex.section import Chapter, Subsection
#from pylatex import Document, Section, Figure, NoEscape, Command

%run ../pipeline.ipynb

import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_columns', None)

In [2]:
coltypes={'ID':str,'TIC':str,'gaiadr3_source_id':str,'epic_id':str,'Score1':'Int64','Flare1':'Int64',
          'Shelf1':'Int64','Evolution1':'Int64','Score2':'Int64','Flare2':'Int64','Shelf2':'Int64','Evolution2':'Int64'}

In [3]:
lctable=pd.read_csv('../lcscoresandchecker.csv',dtype=coltypes)

In [4]:
lctable.drop(columns=["complex_status1", "complex_status2","Notes1","Notes2"], inplace=True)

In [5]:
lctable.head(5)

,pop_id,lcname,idtype,ID,secorcamp,LC_author,cadence,fluxamplitude,TIC,gaiadr3_source_id,epic_id,per,per2,group,age_Myr,disco_paper,paper_author,year,Score1,Flare1,Shelf1,Evolution1,Score2,Flare2,Shelf2,Evolution2
0,0,EPIC-246676629-13-EVEREST-1800,EPIC,246676629,13,EVEREST,1800,0.075449,59129133,3392549449695395968,246676629,0.6253,0.6332,Taurus,2.0,Stauffer_2018,Stauffer,2018,2,0,0,1,1,0,0,0
1,0,EPIC-246676629-13-K2SFF-1800,EPIC,246676629,13,K2SFF,1800,0.083411,59129133,3392549449695395968,246676629,0.6253,0.6332,Taurus,2.0,Stauffer_2018,Stauffer,2018,2,0,0,0,1,0,0,0
2,0,TIC-59129133-5-FFI-30min,TIC,59129133,5,FFI,30min,0.041935,59129133,3392549449695395968,246676629,0.6253,0.6332,Taurus,2.0,Stauffer_2018,Stauffer,2018,2,1,0,1,1,1,0,1
3,0,TIC-59129133-32-FFI-10min,TIC,59129133,32,FFI,10min,0.053793,59129133,3392549449695395968,246676629,0.6253,0.6332,Taurus,2.0,Stauffer_2018,Stauffer,2018,2,0,0,1,2,0,0,0
4,0,TIC-59129133-43-SPOC-120,TIC,59129133,43,SPOC,120,0.053993,59129133,3392549449695395968,246676629,0.6253,0.6332,Taurus,2.0,Stauffer_2018,Stauffer,2018,2,2,1,1,2,2,0,1


In [6]:
#eventually need to add a bit to drop the non-favored light curve for K2 Campaigns

In [7]:
objectswreferences = pd.read_csv("Full_complex_sarah - Objects and References.csv")
references = pd.read_csv("Full_complex_sarah - References.csv")

In [8]:
k2dates=pd.read_csv("K2_Dates.csv")
tessdates=pd.read_csv("TESS_FFI_observation_times.csv")

In [9]:
tessdates['Start Time']=pd.to_datetime(tessdates['Start of Orbit'])
tessdates['End Time']=pd.to_datetime(tessdates['End of Orbit'])
tessdates=tessdates.groupby('Sector').agg({'Start Time':'min','End Time':'max'}) #getting rid of gaps in the sectors
tessdates=tessdates.reset_index()
tessdates['secorcamp']='Sector '+tessdates['Sector'].astype(str)

In [10]:
k2dates=k2dates[['Campaign','Start','Stop']]
k2dates['Start']=pd.to_datetime(k2dates['Start'],format='%Y %b %d')
k2dates['Stop']=pd.to_datetime(k2dates['Stop'],format='%Y %b %d')
k2dates['secorcamp']='Campaign '+k2dates['Campaign'].astype(str)

In [11]:
obstimes=pd.concat([tessdates[['secorcamp','Start Time','End Time']].rename(columns={'Start Time':'Start','End Time':'Stop'}),k2dates[['secorcamp','Start','Stop']]],ignore_index=True)
obstimes['time_diff_days'] = (obstimes['Stop'] - obstimes['Start']).dt.total_seconds() / 86400

In [12]:
obstimes

,secorcamp,Start,Stop,time_diff_days
0,Sector 1,2018-07-25 19:35:00,2018-08-22 16:25:00,27.868056
1,Sector 2,2018-08-23 14:35:00,2018-09-20 00:30:00,27.413194
2,Sector 3,2018-09-21 05:25:00,2018-10-17 21:00:00,26.649306
3,Sector 4,2018-10-19 09:45:00,2018-11-14 08:35:00,25.951389
4,Sector 5,2018-11-15 11:35:00,2018-12-11 19:05:00,26.312500
...,...,...,...,...
114,Campaign 15,2017-08-23 00:00:00,2017-11-20 00:00:00,89.000000
115,Campaign 16,2017-12-07 00:00:00,2018-02-25 00:00:00,80.000000
116,Campaign 17,2018-03-01 00:00:00,2018-05-08 00:00:00,68.000000
117,Campaign 18,2018-05-12 00:00:00,2018-07-02 00:00:00,51.000000


In [13]:
lctable['secorcamp'] = lctable.apply(
    lambda row: f"Sector {row['secorcamp']}" if row['idtype'] == 'TIC' else f"Campaign {row['secorcamp']}", 
    axis=1
)

In [14]:
lctable = lctable.merge(obstimes, on='secorcamp', how='left')

In [15]:
lctable

,pop_id,lcname,idtype,ID,secorcamp,LC_author,cadence,fluxamplitude,TIC,gaiadr3_source_id,epic_id,per,per2,group,age_Myr,disco_paper,paper_author,year,Score1,Flare1,Shelf1,Evolution1,Score2,Flare2,Shelf2,Evolution2,Start,Stop,time_diff_days
0,0,EPIC-246676629-13-EVEREST-1800,EPIC,246676629,Campaign 13,EVEREST,1800,0.075449,59129133,3392549449695395968,246676629,0.6253,0.6332,Taurus,2.0,Stauffer_2018,Stauffer,2018,2,0,0,1,1,0,0,0,2017-03-08 00:00:00,2017-05-27 00:00:00,80.000000
1,0,EPIC-246676629-13-K2SFF-1800,EPIC,246676629,Campaign 13,K2SFF,1800,0.083411,59129133,3392549449695395968,246676629,0.6253,0.6332,Taurus,2.0,Stauffer_2018,Stauffer,2018,2,0,0,0,1,0,0,0,2017-03-08 00:00:00,2017-05-27 00:00:00,80.000000
2,0,TIC-59129133-5-FFI-30min,TIC,59129133,Sector 5,FFI,30min,0.041935,59129133,3392549449695395968,246676629,0.6253,0.6332,Taurus,2.0,Stauffer_2018,Stauffer,2018,2,1,0,1,1,1,0,1,2018-11-15 11:35:00,2018-12-11 19:05:00,26.312500
3,0,TIC-59129133-32-FFI-10min,TIC,59129133,Sector 32,FFI,10min,0.053793,59129133,3392549449695395968,246676629,0.6253,0.6332,Taurus,2.0,Stauffer_2018,Stauffer,2018,2,0,0,1,2,0,0,0,2020-11-19 13:45:00,2020-12-16 17:45:00,27.166667
4,0,TIC-59129133-43-SPOC-120,TIC,59129133,Sector 43,SPOC,120,0.053993,59129133,3392549449695395968,246676629,0.6253,0.6332,Taurus,2.0,Stauffer_2018,Stauffer,2018,2,2,1,1,2,2,0,1,2021-09-16 16:00:00,2021-10-11 09:30:00,24.729167
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
979,211,EPIC-247794636-13-K2SFF-1800,EPIC,247794636,Campaign 13,K2SFF,1800,0.230866,268324578,147799209159857280,247794636,1.7211,2.6225,Taurus,2.0,Rebull_2020,Rebull,2020,9,0,0,0,9,0,0,0,2017-03-08 00:00:00,2017-05-27 00:00:00,80.000000
980,211,TIC-268324578-43-SPOC-120,TIC,268324578,Sector 43,SPOC,120,0.153299,268324578,147799209159857280,247794636,1.7211,2.6225,Taurus,2.0,Rebull_2020,Rebull,2020,2,0,0,1,1,0,0,1,2021-09-16 16:00:00,2021-10-11 09:30:00,24.729167
981,211,TIC-268324578-44-SPOC-120,TIC,268324578,Sector 44,SPOC,120,0.122678,268324578,147799209159857280,247794636,1.7211,2.6225,Taurus,2.0,Rebull_2020,Rebull,2020,0,0,0,0,2,0,0,0,2021-10-12 16:30:00,2021-11-05 22:45:00,24.260417
982,211,TIC-268324578-70-FFI-200s,TIC,268324578,Sector 70,FFI,200s,0.015121,268324578,147799209159857280,247794636,1.7211,2.6225,Taurus,2.0,Rebull_2020,Rebull,2020,2,0,0,0,1,0,0,0,2023-09-20 20:30:00,2023-10-16 08:15:00,25.489583


In [16]:
paper_columns = objectswreferences.columns[objectswreferences.columns.get_loc('Rebull_2016'):]
references_by_name = references.set_index('Name')
objectswreferences_by_tic = objectswreferences.copy()
objectswreferences_by_tic['TIC'] = pd.to_numeric(objectswreferences_by_tic['TIC'], errors='coerce')
objectswreferences_by_tic = objectswreferences_by_tic.dropna(subset=['TIC']).drop_duplicates(subset=['TIC']).set_index('TIC')

def split_camp_or_sec(value):
    if pd.isna(value):
        return None, None
    text = str(value).strip()
    if not text:
        return None, None
    parts = text.rsplit(' ', 1)
    if len(parts) != 2:
        return None, None
    label, number = parts
    try:
        return label.strip().lower(), int(number)
    except ValueError:
        return None, None

def reference_matches(camporsec, first_camp_or_sec, last_camp_or_sec):
    target_label, target_number = split_camp_or_sec(camporsec)
    first_label, first_number = split_camp_or_sec(first_camp_or_sec)
    last_label, last_number = split_camp_or_sec(last_camp_or_sec)

    if None in {target_label, target_number, first_label, first_number, last_label, last_number}:
        return False
    if target_label != first_label or target_label != last_label:
        return False

    return first_number <= target_number <= last_number or last_number <= target_number <= first_number

def compute_seen_count(row):
    tic = pd.to_numeric(row['TIC'], errors='coerce')
    if pd.isna(tic) or tic not in objectswreferences_by_tic.index:
        return 0

    object_row = objectswreferences_by_tic.loc[tic]
    count = 0

    for paper_column in paper_columns:
        paper_value = object_row[paper_column]
        if pd.isna(paper_value) or str(paper_value).strip() == '':
            continue
        if paper_column not in references_by_name.index:
            continue

        reference_row = references_by_name.loc[paper_column]
        if reference_matches(row['secorcamp'], reference_row['First Camp or Sec'], reference_row['Last Camp or Sec']):
            count += 1

    return count

lctable['seen_count'] = lctable.apply(compute_seen_count, axis=1).astype(int)


,pop_id,lcname,idtype,ID,secorcamp,LC_author,cadence,fluxamplitude,TIC,gaiadr3_source_id,epic_id,per,per2,group,age_Myr,disco_paper,paper_author,year,Score1,Flare1,Shelf1,Evolution1,Score2,Flare2,Shelf2,Evolution2,Start,Stop,time_diff_days,seen_count
0,0,EPIC-246676629-13-EVEREST-1800,EPIC,246676629,Campaign 13,EVEREST,1800,0.075449,59129133,3392549449695395968,246676629,0.6253,0.6332,Taurus,2.0,Stauffer_2018,Stauffer,2018,2,0,0,1,1,0,0,0,2017-03-08 00:00:00,2017-05-27 00:00:00,80.000000,2
1,0,EPIC-246676629-13-K2SFF-1800,EPIC,246676629,Campaign 13,K2SFF,1800,0.083411,59129133,3392549449695395968,246676629,0.6253,0.6332,Taurus,2.0,Stauffer_2018,Stauffer,2018,2,0,0,0,1,0,0,0,2017-03-08 00:00:00,2017-05-27 00:00:00,80.000000,2
2,0,TIC-59129133-5-FFI-30min,TIC,59129133,Sector 5,FFI,30min,0.041935,59129133,3392549449695395968,246676629,0.6253,0.6332,Taurus,2.0,Stauffer_2018,Stauffer,2018,2,1,0,1,1,1,0,1,2018-11-15 11:35:00,2018-12-11 19:05:00,26.312500,0
3,0,TIC-59129133-32-FFI-10min,TIC,59129133,Sector 32,FFI,10min,0.053793,59129133,3392549449695395968,246676629,0.6253,0.6332,Taurus,2.0,Stauffer_2018,Stauffer,2018,2,0,0,1,2,0,0,0,2020-11-19 13:45:00,2020-12-16 17:45:00,27.166667,0
4,0,TIC-59129133-43-SPOC-120,TIC,59129133,Sector 43,SPOC,120,0.053993,59129133,3392549449695395968,246676629,0.6253,0.6332,Taurus,2.0,Stauffer_2018,Stauffer,2018,2,2,1,1,2,2,0,1,2021-09-16 16:00:00,2021-10-11 09:30:00,24.729167,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
979,211,EPIC-247794636-13-K2SFF-1800,EPIC,247794636,Campaign 13,K2SFF,1800,0.230866,268324578,147799209159857280,247794636,1.7211,2.6225,Taurus,2.0,Rebull_2020,Rebull,2020,9,0,0,0,9,0,0,0,2017-03-08 00:00:00,2017-05-27 00:00:00,80.000000,1
980,211,TIC-268324578-43-SPOC-120,TIC,268324578,Sector 43,SPOC,120,0.153299,268324578,147799209159857280,247794636,1.7211,2.6225,Taurus,2.0,Rebull_2020,Rebull,2020,2,0,0,1,1,0,0,1,2021-09-16 16:00:00,2021-10-11 09:30:00,24.729167,0
981,211,TIC-268324578-44-SPOC-120,TIC,268324578,Sector 44,SPOC,120,0.122678,268324578,147799209159857280,247794636,1.7211,2.6225,Taurus,2.0,Rebull_2020,Rebull,2020,0,0,0,0,2,0,0,0,2021-10-12 16:30:00,2021-11-05 22:45:00,24.260417,0
982,211,TIC-268324578-70-FFI-200s,TIC,268324578,Sector 70,FFI,200s,0.015121,268324578,147799209159857280,247794636,1.7211,2.6225,Taurus,2.0,Rebull_2020,Rebull,2020,2,0,0,0,1,0,0,0,2023-09-20 20:30:00,2023-10-16 08:15:00,25.489583,0


In [17]:
lctable['seen_before'] = lctable['seen_count'] != 0

In [20]:
lctable[lctable.pop_id==101]

,pop_id,lcname,idtype,ID,secorcamp,LC_author,cadence,fluxamplitude,TIC,gaiadr3_source_id,epic_id,per,per2,group,age_Myr,disco_paper,paper_author,year,Score1,Flare1,Shelf1,Evolution1,Score2,Flare2,Shelf2,Evolution2,Start,Stop,time_diff_days,seen_count,seen_before
424,101,TIC-206544316-1-SPOC-120,TIC,206544316,Sector 1,SPOC,120,0.078336,206544316,4909009394396454144,NaN,0.322083,NaN,Tucana-Horologium,40.0,Zhan_2019,Zhan,2019,2,1,0,1,<NA>,<NA>,<NA>,<NA>,2018-07-25 19:35:00,2018-08-22 16:25:00,27.868056,6,True
425,101,TIC-206544316-2-SPOC-120,TIC,206544316,Sector 2,SPOC,120,0.074438,206544316,4909009394396454144,NaN,0.322083,NaN,Tucana-Horologium,40.0,Zhan_2019,Zhan,2019,2,0,1,1,<NA>,<NA>,<NA>,<NA>,2018-08-23 14:35:00,2018-09-20 00:30:00,27.413194,6,True
426,101,TIC-206544316-28-SPOC-120,TIC,206544316,Sector 28,SPOC,120,0.069069,206544316,4909009394396454144,NaN,0.322083,NaN,Tucana-Horologium,40.0,Zhan_2019,Zhan,2019,2,1,0,0,<NA>,<NA>,<NA>,<NA>,2020-07-31 08:25:00,2020-08-25 14:30:00,25.253472,3,True
427,101,TIC-206544316-29-SPOC-120,TIC,206544316,Sector 29,SPOC,120,0.065397,206544316,4909009394396454144,NaN,0.322083,NaN,Tucana-Horologium,40.0,Zhan_2019,Zhan,2019,2,1,1,1,<NA>,<NA>,<NA>,<NA>,2020-08-26 17:45:00,2020-09-21 22:35:00,26.201389,3,True
428,101,TIC-206544316-68-SPOC-120,TIC,206544316,Sector 68,SPOC,120,0.090977,206544316,4909009394396454144,NaN,0.322083,NaN,Tucana-Horologium,40.0,Zhan_2019,Zhan,2019,2,1,1,1,<NA>,<NA>,<NA>,<NA>,2023-07-29 02:40:00,2023-08-25 15:30:00,27.534722,0,False
429,101,TIC-206544316-69-SPOC-120,TIC,206544316,Sector 69,SPOC,120,0.091282,206544316,4909009394396454144,NaN,0.322083,NaN,Tucana-Horologium,40.0,Zhan_2019,Zhan,2019,2,1,1,1,<NA>,<NA>,<NA>,<NA>,2023-08-25 20:30:00,2023-09-20 15:30:00,25.791667,0,False
430,101,TIC-206544316-95-SPOC-120,TIC,206544316,Sector 95,SPOC,120,0.147959,206544316,4909009394396454144,NaN,0.322083,NaN,Tucana-Horologium,40.0,Zhan_2019,Zhan,2019,2,0,0,0,<NA>,<NA>,<NA>,<NA>,2025-07-25 20:00:00,2025-08-20 01:35:00,25.232639,0,False
431,101,TIC-206544316-96-SPOC-120,TIC,206544316,Sector 96,SPOC,120,0.143528,206544316,4909009394396454144,NaN,0.322083,NaN,Tucana-Horologium,40.0,Zhan_2019,Zhan,2019,2,1,0,0,<NA>,<NA>,<NA>,<NA>,2025-08-20 06:35:00,2025-09-14 22:50:00,25.677083,0,False


In [21]:
lctable.to_csv("papertable5.csv", index=False)